# Laboratorium 7

Celem siódmego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmu głębokiego uczenia aktywnego - Actor-Critic. Zaimplementowany algorytm będzie testowany z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gymnasium as gym
import numpy as np
import random

Dołączenie bibliotek do obsługi sieci neuronowych

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, num_layers=1, learning_rate=0.001):
        super(DQN, self).__init__()

        self.entrance = nn.Linear(state_size, hidden_neurons)
        for i in range(num_layers - 1):
             setattr(self, f'fc{i+1}', nn.Linear(hidden_neurons, hidden_neurons))
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.entrance(x))
        for i in range(len(self._modules) - 2):
            x = F.relu(getattr(self, f'fc{i+1}')(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = np.array(states, dtype=np.float32)
        targets = np.array(targets, dtype=np.float32)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
        self.optimizer.step()

## Zadanie 1 - Actor-Critic

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Actor-Critic. W tym celu należy utworzyć dwie głębokie sieci neuronowe:
    1. *actor* - sieć, która będzie uczyła się optymalnej strategii (podobna do tej z laboratorium 6),
    2. *critic* - sieć, która będzie uczyła się funkcji oceny stanu (podobnie jak się DQN).
Wagi sieci *actor* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    \theta \leftarrow \theta + \alpha \delta_t \nabla_\theta log \pi_{\theta}(a_t, s_t | \theta).
\end{equation*}
Wagi sieci *critic* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    w \leftarrow w + \beta \delta_t \nabla_w\upsilon(s_{t + 1}, w),
\end{equation*}
gdzie:
\begin{equation*}
    \delta_t \leftarrow r_t + \gamma \upsilon(s_{t + 1}, w) - \upsilon(s_t, w).
\end{equation*}
</p>

In [3]:
class REINFORCEAgent:
    def __init__(self, state_size, action_size, actor, critic):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.99    # discount rate
        self.learning_rate = 0.001
        self.actor = actor
        self.critic = critic #critic network should have only one output


    def get_action(self, state):
        """
        Compute the action to take in the current state, basing on policy returned by the network.

        Note: To pick action according to the probability generated by the network
        """      

        q_values = self.actor.predict(state)
        # Add numerical stability: subtract max before exp
        q_values = q_values - np.max(q_values)
        action_probabilities = np.exp(q_values) / np.sum(np.exp(q_values))
        chosen_action = np.random.choice(self.action_size, p=action_probabilities)
        
        return chosen_action

  

    def learn(self, state, action, reward, next_state, done):
        """
        Function learn networks using information about state, action, reward and next state. 
        First the values for state and next_state should be estimated based on output of critic network.
        Critic network should be trained based on target value:
        target = r + \gamma next_state_value if not done]
        target = r if done.
        Actor network shpuld be trained based on delta value:
        delta = target - state_value
        """
        state_value = self.critic.predict(state)
        next_state_value = self.critic.predict(next_state)

        target = reward + self.gamma * next_state_value * (1 - int(done))
        delta = target - state_value

        # Update critic network
        self.critic.fit(state, target)

        # Update actor network
        # Actor should learn to output delta for the taken action
        actor_targets = np.zeros(self.action_size)
        actor_targets[action] = delta
        self.actor.fit(state, actor_targets)
        

<>:33: SyntaxWarning: invalid escape sequence '\g'
<>:33: SyntaxWarning: invalid escape sequence '\g'
C:\Users\Filip\AppData\Local\Temp\ipykernel_3168\544882983.py:33: SyntaxWarning: invalid escape sequence '\g'
  target = r + \gamma next_state_value if not done]


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [4]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
alpha_learning_rate = 0.0001
beta_learning_rate = 0.0005

print(f"State size: {state_size}, Action size: {action_size}")

actor_model = DQN(state_size, action_size, hidden_neurons=128, num_layers=2, learning_rate=alpha_learning_rate)

critic_model = DQN(state_size, 1, hidden_neurons=128, num_layers=2, learning_rate=beta_learning_rate)

c:\Users\Filip\Documents\mgr-siium\guzw\.venv\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


State size: 4, Action size: 2


Czas nauczyć agenta gry w środowisku *CartPool*:

In [5]:
agent = REINFORCEAgent(state_size, action_size, actor_model, critic_model)


for i in range(100):
    score_history = []

    for i in range(100):
        done = False
        score = 0
        state = env.reset()[0]
        while not done:
            action = agent.get_action(state)
            next_state, reward, done, _, _ = env.step(action)
            agent.learn(state, action, reward, next_state, done)
            state = next_state
            score += reward
        score_history.append(score)

    print("mean reward:%.3f" % (np.mean(score_history)))

    if np.mean(score_history) > 300:
        print("You Win!")
        break

C:\Users\Filip\AppData\Local\Temp\ipykernel_3168\544882983.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  actor_targets[action] = delta


mean reward:12.850
mean reward:9.450
mean reward:9.390
mean reward:9.320


KeyboardInterrupt: 